# 🚀 OmniParser Full GPU Server trên Google Colab (100% Original Pipeline)

Notebook này khởi chạy **100% Pipeline gốc của Microsoft OmniParser** (YOLOv8 Icon Detection + Florence-2 Full Semantics Captioning + EasyOCR) trên GPU miễn phí của Google Colab (Tesla T4) và xuất ra đường link công khai (`https://xxxx.trycloudflare.com`) để kết nối về máy tính Linux của bạn.

### Hướng dẫn sử dụng:
1. Trên thanh menu Colab, chọn **Runtime -> Change runtime type -> T4 GPU**.
2. Bấm **Runtime -> Run all** (hoặc chạy lần lượt 3 bước bên dưới).
3. Copy đường link `https://xxxx.trycloudflare.com` xuất hiện ở Bước 3 và dán vào lệnh chạy trên máy Linux của bạn.

In [ ]:
# @title Bước 1: Kiểm tra GPU & Cài đặt thư viện + Tải Model Weights gốc
!nvidia-smi

!git clone https://github.com/doantrongthai/fix_omni_parser.git /content/OmniParser
%cd /content/OmniParser

# Cài đặt các thư viện cần thiết cho pipeline
!pip install -q ultralytics supervision==0.18.0 easyocr transformers==4.46.3 accelerate timm einops fastapi uvicorn huggingface_hub tabulate

# Tải bộ trọng số gốc của Microsoft OmniParser v2.0 (Icon Detect + Florence-2 Caption)
from huggingface_hub import snapshot_download
import os, shutil

os.makedirs("/content/OmniParser/weights", exist_ok=True)
snapshot_download(repo_id="microsoft/OmniParser-v2.0", local_dir="/content/OmniParser/weights")
if os.path.exists("/content/OmniParser/weights/icon_caption") and not os.path.exists("/content/OmniParser/weights/icon_caption_florence"):
    shutil.move("/content/OmniParser/weights/icon_caption", "/content/OmniParser/weights/icon_caption_florence")

# Tải công cụ Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt xong toàn bộ môi trường và trọng số gốc!")

In [ ]:
# @title Bước 2: Khởi tạo OmniParser Engine trên GPU CUDA (Có Trace Log & Latency Breakdown)
import sys, time, io, base64, torch, cv2
import numpy as np
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
from torchvision.ops import box_convert

sys.path.insert(0, "/content/OmniParser")
from util.utils import (
    get_caption_model_processor, get_yolo_model, check_ocr_box,
    predict_yolo, remove_overlap_new, int_box_area, get_parsed_content_icon, annotate
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Đang nạp mô hình lên thiết bị: {device.upper()}...")

som_model = get_yolo_model(model_path="/content/OmniParser/weights/icon_detect/model.pt")
caption_model_processor = get_caption_model_processor(
    model_name="florence2",
    model_name_or_path="/content/OmniParser/weights/icon_caption_florence",
    device=device
)

app = FastAPI()

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class ParseRequest(BaseModel):
    base64_image: str
    use_local_semantics: bool = True

@app.post("/parse")
@app.post("/parse/")
def parse(req: ParseRequest):
    req_start = time.perf_counter()
    image_bytes = base64.b64decode(req.base64_image)
    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    w, h = image.size

    print("\n" + "="*65)
    print(f"📸 [REQUEST MỚI] Nhận ảnh màn hình: {w}x{h} px")
    print("="*65)

    # 1. Đo tiền xử lý cấu hình vẽ bbox
    sync_cuda()
    t0 = time.perf_counter()
    box_overlay_ratio = max(w, h) / 3200
    draw_bbox_config = {
        "text_scale": 0.8 * box_overlay_ratio,
        "text_thickness": max(int(2 * box_overlay_ratio), 1),
        "text_padding": max(int(3 * box_overlay_ratio), 1),
        "thickness": max(int(3 * box_overlay_ratio), 1),
    }
    t_pre = (time.perf_counter() - t0) * 1000

    # 2. Bước OCR Text
    sync_cuda()
    t1 = time.perf_counter()
    (ocr_texts, ocr_bboxes), _ = check_ocr_box(
        image, display_img=False, output_bb_format="xyxy",
        easyocr_args={"text_threshold": 0.8}, use_paddleocr=False
    )
    sync_cuda()
    t_ocr = (time.perf_counter() - t1) * 1000
    print(f"📝 [1. OCR Text]           : {t_ocr:7.2f} ms | Tìm thấy {len(ocr_texts)} text boxes")

    # 3. Bước YOLO Icon Detect
    sync_cuda()
    t2 = time.perf_counter()
    xyxy, logits, phrases = predict_yolo(
        model=som_model, image=image, box_threshold=0.05,
        imgsz=(h, w), scale_img=False, iou_threshold=0.1
    )
    sync_cuda()
    t_yolo = (time.perf_counter() - t2) * 1000
    print(f"🎯 [2. YOLO Icon]          : {t_yolo:7.2f} ms | Phát hiện {len(xyxy)} raw icons")

    # 4. Bước Lọc trùng lặp & Khớp OCR vào Icon
    sync_cuda()
    t3 = time.perf_counter()
    xyxy_norm = xyxy / torch.Tensor([w, h, w, h]).to(xyxy.device)
    xyxy_elem = [
        {"type": "icon", "bbox": b, "interactivity": True, "content": None}
        for b in xyxy_norm.tolist() if int_box_area(b, w, h) > 0
    ]
    if ocr_bboxes:
        ocr_bbox_norm = (torch.tensor(ocr_bboxes) / torch.Tensor([w, h, w, h])).tolist()
        ocr_bbox_elem = [
            {"type": "text", "bbox": b, "interactivity": False, "content": txt, "source": "box_ocr_content_ocr"}
            for b, txt in zip(ocr_bbox_norm, ocr_texts) if int_box_area(b, w, h) > 0
        ]
    else:
        ocr_bbox_elem = []

    filtered_boxes = remove_overlap_new(boxes=xyxy_elem, iou_threshold=0.7, ocr_bbox=ocr_bbox_elem)
    filtered_boxes_elem = sorted(filtered_boxes, key=lambda x: x["content"] is None)
    starting_idx = next((i for i, b in enumerate(filtered_boxes_elem) if b["content"] is None), -1)
    filtered_boxes_tensor = torch.tensor([b["bbox"] for b in filtered_boxes_elem])
    num_caption_needed = len(filtered_boxes_elem) - starting_idx if starting_idx != -1 else 0
    sync_cuda()
    t_filter = (time.perf_counter() - t3) * 1000
    print(f"✂️ [3. Overlap Filter]    : {t_filter:7.2f} ms | Còn {len(filtered_boxes_elem)} boxes ({num_caption_needed} cần caption)")

    # 5. Bước Florence-2 Captioning
    sync_cuda()
    t4 = time.perf_counter()
    image_np = np.asarray(image)
    if req.use_local_semantics and num_caption_needed > 0:
        parsed_content_icon = get_parsed_content_icon(
            filtered_boxes_tensor, starting_idx, image_np,
            caption_model_processor, prompt="<CAPTION>", batch_size=128
        )
        for box in filtered_boxes_elem:
            if box["content"] is None:
                box["content"] = parsed_content_icon.pop(0) if parsed_content_icon else "icon"
    else:
        for box in filtered_boxes_elem:
            if box["content"] is None:
                box["content"] = "interactive button"
    sync_cuda()
    t_caption = (time.perf_counter() - t4) * 1000
    print(f"🧠 [4. Florence-2 Caption] : {t_caption:7.2f} ms | Đã gán nhãn {num_caption_needed} icons")

    # 6. Bước Vẽ SOM Annotation
    sync_cuda()
    t5 = time.perf_counter()
    boxes_cxcywh = box_convert(boxes=filtered_boxes_tensor, in_fmt="xyxy", out_fmt="cxcywh")
    phrases = [str(i) for i in range(len(boxes_cxcywh))]
    annotated_frame, label_coords = annotate(
        image_source=image_np, boxes=boxes_cxcywh,
        logits=logits[:len(boxes_cxcywh)], phrases=phrases, **draw_bbox_config
    )
    buffered = io.BytesIO()
    Image.fromarray(annotated_frame).save(buffered, format="PNG")
    encoded_image = base64.b64encode(buffered.getvalue()).decode("ascii")
    sync_cuda()
    t_render = (time.perf_counter() - t5) * 1000
    print(f"🎨 [5. SOM Annotation]     : {t_render:7.2f} ms | Vẽ & Encode Base64")

    total_latency = (time.perf_counter() - req_start) * 1000
    print("-" * 65)
    print(f"⏱️ [TỔNG ĐỘ TRỄ LATENCY]   : {total_latency:7.2f} ms ({total_latency/1000:.2f} giây)")
    print("=" * 65 + "\n")

    return {
        "som_image_base64": encoded_image,
        "parsed_content_list": filtered_boxes_elem,
        "latency": total_latency / 1000.0,
        "latency_breakdown_ms": {
            "ocr_text": round(t_ocr, 2),
            "yolo_icon": round(t_yolo, 2),
            "overlap_filter": round(t_filter, 2),
            "florence_caption": round(t_caption, 2),
            "som_annotation": round(t_render, 2)
        }
    }

@app.get("/probe")
@app.get("/probe/")
def probe():
    return {"message": "Omniparser GPU API ready (Detailed Trace Log & Latency Enabled)"}

print("🎉 OmniParser GPU Engine đã khởi tạo thành công với hệ thống Trace Log!")

In [ ]:
# @title Bước 3: Mở Cloudflare Tunnel & Chạy Server (Copy URL bên dưới về máy Linux)
import subprocess, threading, re, time

def start_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            print('\n' + '='*75)
            print(f'🌐 LINK PUBLIC COLAB GPU CỦA BẠN: {public_url}')
            print('='*75)
            print('\n👉 Copy lệnh sau chạy ở Terminal 3 trên máy Linux của bạn:')
            print(f'python app.py --windows_host_url localhost:5000 --omniparser_server_url {public_url}\n')
            print('='*75 + '\n')

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(3)
uvicorn.run(app, host='127.0.0.1', port=8000)